# DroneRL — Research Analysis Notebook

This notebook reproduces the experimental evidence behind the claims in
[`docs/assignment-2/EXPERIMENTS.md`](../docs/assignment-2/EXPERIMENTS.md)
and the cost figures in
[`docs/assignment-2/COST_ANALYSIS.md`](../docs/assignment-2/COST_ANALYSIS.md).

It satisfies the course-wide submission guidelines (§9.2 *Results
Analysis Notebook*) by running the analysis end-to-end inside a
notebook, displaying every chart inline, and surfacing every measured
number from a committed script.

**Running:** `uv run jupyter notebook notebooks/research_analysis.ipynb`
or open it in VS Code with the Python kernel.

---

## 1. Setup — repo root on the Python path

The analysis modules are not yet pip-installed, so put the repo root
on `sys.path` so `import analysis` works from the `notebooks/`
directory.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
    sys.path.insert(0, str(ROOT))

# Run cells with the project root as cwd so relative paths
# (e.g. load_config("config/config.yaml") in the analysis scripts) resolve.
os.chdir(ROOT)

print(f'repo root: {ROOT}')
print(f'cwd: {Path.cwd()}')
print(f'analysis on path: {(ROOT / "analysis").exists()}')

## 2. Multi-seed robustness

5 seeds × 1500 episodes × 3 algorithms on the medium-difficulty board.
Output is a chart with mean ± 95 % CI bands per algorithm.

**Hypothesis (H1):** Bellman fails under noise — its last-200 mean
should be clearly worse than Q-Learning and Double-Q at noise = 0.5.

**What we actually find** (see EXPERIMENTS.md §H1): Bellman ties
Q-Learning at this difficulty; the surprise is that **Double-Q is the
seed-sensitive one** at 1500 episodes — 2 of 5 seeds collapse.

In [ ]:
from analysis.multi_seed_robustness import main as run_multi_seed
run_multi_seed()

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'results' / 'analysis' / 'multi_seed_robustness.png')))

## 3. Alpha-decay sensitivity sweep

6 decay values × 3 seeds × {Q-Learning, Double-Q}, with Bellman as a
horizontal reference line. Tests whether the conclusions about
Q-Learning vs. Bellman are robust to the precise decay value.

**Hypothesis (H2):** Q-Learning's advantage should hold across a
range of decay rates.

**What we actually find** (see EXPERIMENTS.md §H2): Q-Learning is
flat across the entire decay grid (76 → 78). At this difficulty and
training budget, the decay parameter barely matters.

In [ ]:
from analysis.alpha_decay_sweep import main as run_decay_sweep
run_decay_sweep()

In [ ]:
display(Image(filename=str(ROOT / 'results' / 'analysis' / 'alpha_decay_sweep.png')))

## 4. Cost profile

Wall-time, peak heap, and Q-table bytes for each algorithm.
Underpins the runtime cost story in COST_ANALYSIS.md §1–3.

In [ ]:
from analysis.cost_profile import main as run_cost_profile
run_cost_profile()

In [ ]:
import json
with open(ROOT / 'results' / 'analysis' / 'cost_profile.json') as f:
    cost = json.load(f)
for p in cost['profiles']:
    print(f"  {p['algorithm']:12s}  {p['elapsed_s']:6.3f} s  "
          f"{p['episodes_per_s']:6.1f} ep/s  "
          f"{p['us_per_episode']:6.1f} us/ep  "
          f"Q-table {p['q_table_bytes']} bytes")

## 5. Q-table heatmap — what the agent actually learned

The GUI renders a live heatmap during training, but for a *result*
artefact we want a saved PNG matching §9.3's heatmap requirement.
Here we train one Q-Learning run on the medium-difficulty board,
take the max-Q value per cell, and save the heatmap to
`results/analysis/q_table_heatmap.png`.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

from analysis._runner import base_raw_config, with_overrides
from dronerl.agent_factory import create_agent
from dronerl.environment import Environment
from dronerl.hazard_generator import HazardGenerator
from dronerl.trainer import Trainer

raw = base_raw_config()
raw["dynamic_board"]["enabled"] = True
cfg = with_overrides(
    raw, algorithm="q_learning", seed=11,
    noise_level=0.5, hazard_density=0.12, difficulty=0.3,
)
random.seed(11); np.random.seed(11)
env = Environment(cfg)
HazardGenerator(cfg).apply(env)
env.drift_probability = (
    cfg.wind.drift_probability
    * cfg.dynamic_board.noise_level
    * (1.0 + cfg.dynamic_board.difficulty)
)
agent = create_agent(cfg)
trainer = Trainer(agent, env, cfg)
for _ in range(1500):
    trainer.run_episode()

max_q = agent.q_table.max(axis=2)
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(max_q, cmap="viridis", origin="upper")
ax.set_title("Q-Learning — max Q-value per cell after 1 500 episodes\n(medium board, seed=11)")
ax.set_xlabel("col"); ax.set_ylabel("row")
fig.colorbar(im, ax=ax, label="max Q")
out = ROOT / "results" / "analysis" / "q_table_heatmap.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()
print(f"saved: {out}")

## 6. Per-seed convergence scatter — final reward vs. learning speed

Standalone scatter chart (§9.3 / §20.5.b — *Scatter plots*). For each
`(algorithm, seed)` cell we plot:

- **x-axis:** episodes-to-half-target — how fast the algorithm reached
  half of its eventual last-50 mean reward (a rough learning-speed
  proxy).
- **y-axis:** last-50 mean reward — how well it ended up.

A real correlation: faster-learning runs should also reach higher
final reward, except where exploration noise dominates. The expected
shape is a tight cluster for Bellman + Q-Learning (which converge
similarly at medium difficulty) and a more dispersed cluster for
Double-Q (which is bimodal at this training budget — see §H1 in
EXPERIMENTS.md).

Saves to `results/analysis/convergence_scatter.png`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from analysis._runner import base_raw_config, train_cells
from dronerl.comparison import ALGORITHM_COLORS, ALGORITHM_LABELS

raw = base_raw_config()
raw["dynamic_board"]["enabled"] = True
board = {"noise_level": 0.5, "hazard_density": 0.12, "difficulty": 0.3}
seeds = (3, 7, 11, 17, 23)
algos = ("bellman", "q_learning", "double_q")
cells = [(raw, a, s, 1500, board) for a in algos for s in seeds]
results = train_cells(cells, n_workers=1)

fig, ax = plt.subplots(figsize=(8, 6))
for algo in algos:
    xs, ys = [], []
    for a, s, rewards, _ in results:
        if a != algo:
            continue
        last50 = float(np.mean(rewards[-50:]))
        target = last50 / 2.0
        running = []
        episodes_to_half = len(rewards)
        for i in range(50, len(rewards) + 1):
            running_mean = float(np.mean(rewards[max(0, i - 50):i]))
            if running_mean >= target:
                episodes_to_half = i
                break
        xs.append(episodes_to_half)
        ys.append(last50)
    ax.scatter(xs, ys, s=80, c=ALGORITHM_COLORS[algo], label=ALGORITHM_LABELS[algo],
               alpha=0.7, edgecolors="#222", linewidths=0.8)

ax.set_xlabel("Episodes to reach half of final last-50 mean (lower = faster learning)")
ax.set_ylabel("Last-50 mean reward (higher = better final performance)")
ax.set_title("Per-seed convergence scatter — speed vs. quality\n"
             "(5 seeds × 3 algos × 1500 episodes, medium board)")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left", framealpha=0.92)

out = ROOT / "results" / "analysis" / "convergence_scatter.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()
print(f"saved: {out}")

## 5. Conclusions

The qualitative picture in the README's *Conclusions* section holds,
but with two caveats this notebook surfaces directly:

1. **Double-Q is seed-sensitive at short training budgets.** Its
   bias-correction benefit only emerges with enough samples to
   dominate the higher initial variance from running two interleaved
   Q-tables. At Scenario 2's 6 000-episode budget the textbook
   ordering reappears.
2. **Q-Learning's α-decay barely moves the needle** at this difficulty.
   The decay separates Q-Learning from Bellman *in theory* (Σα² < ∞);
   *in practice* the same plateau is reached even at decay = 1.0.

Full hypothesis-by-hypothesis write-up:
[`docs/assignment-2/EXPERIMENTS.md`](../docs/assignment-2/EXPERIMENTS.md).

Cost analysis (runtime + AI-development cost):
[`docs/assignment-2/COST_ANALYSIS.md`](../docs/assignment-2/COST_ANALYSIS.md).